<a href="https://colab.research.google.com/github/Kanomthai4869/AI-SUPERMARKET/blob/main/train_rgb_weight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q tensorflow pandas

import tensorflow as tf
import pandas as pd
import numpy as np
import os
from tensorflow.keras import layers, models

In [ ]:
from google.colab import files
uploaded = files.upload()

🟩 CELL 2: อัปโหลด dataset
dataset.zip
labels.csv
dataset/
 ├── apple/
 │    ├── img1.jpg
 │    ├── img2.jpg
 │
 ├── orange/
 │    ├── img3.jpg
 │    ├── img4.jpg

 labels.csv
filename,weight,label
img1.jpg,120,apple
img2.jpg,130,apple
img3.jpg,200,orange
img4.jpg,210,orange

🟩 CELL 3: แตก zip


In [ ]:
import zipfile

with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

🟩 CELL 4: โหลดข้อมูล

In [ ]:
IMG_SIZE = 224

df = pd.read_csv("labels.csv")

def load_data(df, base_path="dataset"):
    images, weights, labels = [], [], []

    for _, row in df.iterrows():
        path = os.path.join(base_path, row['label'], row['filename'])

        img = tf.keras.preprocessing.image.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
        img = tf.keras.preprocessing.image.img_to_array(img)
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)

        images.append(img)

        # normalize น้ำหนัก (สำคัญ)
        weights.append([row['weight'] / 300.0])

        labels.append(0 if row['label'] == 'apple' else 1)

    return np.array(images), np.array(weights), np.array(labels)

X_img, X_weight, y = load_data(df)

print("Loaded:", X_img.shape, X_weight.shape, y.shape)

🟩 CELL 5: โหลดโมเดล (Transfer Learning)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# freeze
for layer in base_model.layers:
    layer.trainable = False

🟩 CELL 6: สร้างโมเดล (multi-input)

In [ ]:
# image branch
image_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(image_input, training=False)
x = layers.GlobalAveragePooling2D()(x)

# weight branch
weight_input = layers.Input(shape=(1,))
w = layers.Dense(16, activation='relu')(weight_input)

# combine
combined = layers.concatenate([x, w])
z = layers.Dense(64, activation='relu')(combined)
z = layers.Dropout(0.3)(z)
z = layers.Dense(1, activation='sigmoid')(z)

model = models.Model(inputs=[image_input, weight_input], outputs=z)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

🟩 CELL 7: Train

In [ ]:
history = model.fit(
    [X_img, X_weight],
    y,
    epochs=5,
    batch_size=8,
    validation_split=0.2
)

🟩 CELL 8: Save โมเดล

In [ ]:
model.save("fruit_model_transfer.h5")

🟩 CELL 9: ดาวน์โหลดโมเดล

In [ ]:
from google.colab import files
files.download("fruit_model_transfer.h5")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save('/content/drive/MyDrive/fruit_model_transfer.h5')